# DAD pipeline: three conditions
| Condition | SLM stage | What the code LLM sees |
|---|---|---|
| **baseline** | none | requirements + Mermaid diagram |
| **A: fine-tuned + skills** | LoRA adapters (`design_smell_lora`, `code_smell_lora`) via MCP, DAD built with the skill tables | + DAD |
| **B: untrained + skills** | same base Qwen2.5-1.5B with adapters **disabled**, skill text injected as the prompt, via MCP, DAD built with the skill tables | + DAD |

A vs B isolates the effect of fine-tuning, B vs baseline isolates the effect of a skill-prompted SLM, A vs baseline is the proposal's *design-time advisory* arm. The in-loop and iterative-LLM arms from the proposal are **not** in this notebook.

Upload: `design_smell_lora.zip`, `code_smell_lora.zip`, `design_skill.md`, `code_skill.md`. Runtime: T4 GPU. Colab secret `MidSub` = Gemini key.

In [1]:
%%capture
!pip install -q "mcp<2" peft bitsandbytes accelerate google-genai pandas networkx pylint tabulate

In [2]:
import os, re, json, time, ast, shutil, glob, sys, threading, subprocess
import pandas as pd, networkx as nx

# ---------------- config ----------------
STUB    = False      # True = fake SLM outputs, for a GPU-free plumbing test only
N_RUNS  = 2          # generations per (project, condition)
MODEL   = "gemini-3.1-flash-lite"
ART     = "/content/artifacts" if os.path.isdir("/content") else "artifacts"
os.makedirs(ART, exist_ok=True)

NEEDED = ["design_smell_lora.zip", "code_smell_lora.zip", "design_skill.md", "code_skill.md"]
def find_ci(name): return next((f for f in os.listdir(".") if f.lower() == name.lower()), None)
missing = [n for n in NEEDED if not find_ci(n)]
if missing and not STUB:
    from google.colab import files
    print("Please upload:", missing); files.upload()
for n in NEEDED:                                   # normalise file names, unpack adapters
    f = find_ci(n)
    if f and f != n: shutil.move(f, n)
for z in ("design_smell_lora", "code_smell_lora"):
    if os.path.exists(z + ".zip") and not os.path.isdir(z): shutil.unpack_archive(z + ".zip", ".")
    if not STUB: assert glob.glob(f"{z}/adapter_config.json"), f"{z}/adapter_config.json not found (zip nested wrongly?)"
print("Inputs ready.")

Inputs ready.


## 1. Parse the skills (they drive the DAD table, the fix text and the severity rules)

In [3]:
def read(p): return open(p, encoding="utf-8").read()
DESIGN_SKILL, CODE_SKILL = read("design_skill.md"), read("code_skill.md")

def section(md_text, heading):
    m = re.search(rf"^## {re.escape(heading)}\s*\n(.*?)(?=^## |\Z)", md_text, re.S | re.M)
    return m.group(1).strip() if m else ""

def parse_tables(md_text, heading):
    ref, unseen, sub = {}, set(), ""
    for line in section(md_text, heading).splitlines():
        if line.startswith("###"): sub = line.lower(); continue
        cells = [c.strip().strip("*") for c in line.strip().strip("|").split("|")]
        if len(cells) >= 3 and cells[0] and not set(cells[0]) <= set("-: ") and cells[0] != "Smell":
            ref[cells[0]] = (cells[1], cells[2])
            if "unseen" in sub: unseen.add(cells[0])
    return ref, unseen

DESIGN_REF, DESIGN_UNSEEN = parse_tables(DESIGN_SKILL, "Smells and standard fixes")
CODE_REF,   CODE_UNSEEN   = parse_tables(CODE_SKILL,   "Smells and standard fixes")
CODE_SEV = {"Long Parameter List": "medium", "Long Method": "medium",
            "Primitive Obsession": "low", "Magic Number Risk": "low", "Duplicate Code": "low"}   # rule from code_skill.md
REF   = {"design": DESIGN_REF,  "code": CODE_REF}
UNSEEN= {"design": DESIGN_UNSEEN, "code": CODE_UNSEEN}
assert len(DESIGN_REF) == 9 and len(CODE_REF) == 5, (len(DESIGN_REF), len(CODE_REF))   # 6+3 and 4+1
print("design:", list(DESIGN_REF)); print("code:", list(CODE_REF)); print("unseen:", DESIGN_UNSEEN | CODE_UNSEEN)

design: ['God Class', 'Cyclic Dependency', 'Hub-like Modularization', 'Deep Hierarchy', 'Wide Hierarchy', 'Data Class', 'Middle Man', 'Multifaceted Abstraction', 'Insufficient Modularization']
code: ['Long Parameter List', 'Primitive Obsession', 'Long Method', 'Magic Number Risk', 'Duplicate Code']
unseen: {'Middle Man', 'Duplicate Code', 'Insufficient Modularization', 'Multifaceted Abstraction'}


## 2. MCP server `smell-advisor`
One base model, two LoRA adapters swapped with `set_adapter`, and `disable_adapter()` for the untrained condition. Tool names match the skills (`detect_design_smells`, `detect_code_smells`, `read_design_doc`). Fine-tuned mode uses the **exact training system prompts**.

In [4]:
%%writefile mcp_server.py
"""smell-advisor MCP server: one 4-bit Qwen2.5-1.5B base, two LoRA adapters, plus a 'base' (no adapter) mode."""
import os, re, json
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("smell-advisor")
STUB = os.environ.get("SMELL_STUB") == "1"          # plumbing smoke-test without a GPU
BASE = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
ADAPTERS = {"design": "design_smell_lora", "code": "code_smell_lora"}
ADAPTER_CTX = {"design": 3072, "code": 2048}        # max_seq_length each adapter was trained with
BASE_CTX = 6144
DOC_ROOT = "designs"

# EXACT system prompts used at training time (the adapters expect these).
DESIGN_SYSTEM = (
    "You detect design smells in UML class diagrams written in Mermaid. "
    "Possible smells: God Class, Cyclic Dependency, Hub-like Modularization, "
    "Deep Hierarchy, Wide Hierarchy, Data Class. "
    "Output only a JSON array of objects with keys 'smell', 'class', 'severity' "
    "(high/medium/low). Output [] if there are no smells.")
CODE_SYSTEM = (
    "You predict potential code smells from UML class diagrams written in Mermaid. "
    "Look at the method signatures and attributes. "
    "Possible smells: Long Parameter List (6+ args), Primitive Obsession (raw String/int for email, phone, zip, currency), "
    "Long Method (names combining 3+ verbs), Magic Number Risk (raw numbers for taxRate, discount, retry, timeout). "
    "Output only a JSON array of objects with keys 'smell', 'class', 'method' (or 'attribute' if no method), 'reason'. "
    "Output [] if there are no potential code smells.")
PROBE = {  # appended only when probe=True, to test generalisation to smells the adapters never saw
    "design": " Also consider these additional smells: Middle Man, Multifaceted Abstraction, Insufficient Modularization.",
    "code": " Also consider this additional smell: Duplicate Code."}
SKILL_FILES = {"design": "design_skill.md", "code": "code_skill.md"}

_model = _tok = None
_skill_cache = {}

def _section(md, heading):
    m = re.search(rf"^## {re.escape(heading)}\s*\n(.*?)(?=^## |\Z)", md, re.S | re.M)
    return m.group(0).strip() if m else ""

def skill_system(kind):
    """System prompt for the UNTRAINED model: the skill's smell table + output format, nothing else."""
    if kind not in _skill_cache:
        md = open(SKILL_FILES[kind], encoding="utf-8").read()
        what = "design smells" if kind == "design" else "potential code smells"
        _skill_cache[kind] = (
            f"You are a tool that detects {what} in a Mermaid classDiagram. Follow this skill specification.\n\n"
            + re.sub(r"^###.*\n", "", _section(md, "Smells and standard fixes"), flags=re.M) + "\n\n" + _section(md, "Output format")
            + "\n\nYou cannot call tools. Analyse the diagram in the user message and reply with ONLY the JSON array "
              "described under 'Output format': no prose, no markdown fences. Report only smells visible in the diagram. "
              "Use [] if there are none.")
    return _skill_cache[kind]

def _load():
    global _model, _tok
    if _model is not None: return
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from peft import PeftModel
    _tok = AutoTokenizer.from_pretrained(BASE); _tok.padding_side = "left"
    base = AutoModelForCausalLM.from_pretrained(BASE, device_map={"": 0})   # bnb-4bit config ships with the checkpoint
    m = PeftModel.from_pretrained(base, ADAPTERS["design"], adapter_name="design")
    m.load_adapter(ADAPTERS["code"], adapter_name="code")
    _model = m.eval()

def _run(kind, diagram, mode, probe, retry):
    if mode not in ("finetuned", "base"): return "ERROR: mode must be 'finetuned' or 'base'"
    if STUB:
        return ('[{"smell":"God Class","class":"StoreManager","severity":"high"}]' if kind == "design" and "StoreManager" in diagram
                else "[]")
    import torch
    _load()
    if mode == "finetuned":
        system = (DESIGN_SYSTEM if kind == "design" else CODE_SYSTEM) + (PROBE[kind] if probe else "")
    else:
        system = skill_system(kind)
    prompt = _tok.apply_chat_template([{"role": "system", "content": system}, {"role": "user", "content": diagram}],
                                      tokenize=False, add_generation_prompt=True)
    enc = _tok(prompt, return_tensors="pt").to("cuda")
    limit = ADAPTER_CTX[kind] if mode == "finetuned" else BASE_CTX
    if enc.input_ids.shape[1] + 512 > limit:
        return f"ERROR: diagram too long for {limit}-token context; split it"
    kw = dict(max_new_tokens=512, do_sample=bool(retry))
    if retry: kw.update(temperature=0.3, top_p=0.9)   # greedy retry would just repeat itself
    with torch.no_grad():
        if mode == "finetuned":
            _model.set_adapter(kind)
            out = _model.generate(**enc, **kw)
        else:
            with _model.disable_adapter():
                out = _model.generate(**enc, **kw)
    return _tok.decode(out[0, enc.input_ids.shape[1]:], skip_special_tokens=True).strip()

@mcp.tool()
def detect_design_smells(diagram: str, mode: str = "finetuned", probe: bool = False, retry: bool = False) -> str:
    """Detect design smells in a Mermaid classDiagram. mode='finetuned' uses the design LoRA adapter; mode='base' uses the untrained model prompted with the skill. Returns a JSON array."""
    return _run("design", diagram, mode, probe, retry)

@mcp.tool()
def detect_code_smells(diagram: str, mode: str = "finetuned", probe: bool = False, retry: bool = False) -> str:
    """Predict code smells from a Mermaid classDiagram. mode='finetuned' uses the code LoRA adapter; mode='base' uses the untrained model prompted with the skill. Returns a JSON array."""
    return _run("code", diagram, mode, probe, retry)

@mcp.tool()
def read_design_doc(path: str) -> str:
    """Read a design document (.mmd/.md) from the designs/ folder and return its Mermaid classDiagram text."""
    p, root = os.path.realpath(path), os.path.realpath(DOC_ROOT)
    if not p.startswith(root + os.sep): return f"ERROR: only files under {DOC_ROOT}/ may be read"
    text = open(p, encoding="utf-8").read()
    m = re.search(r"```mermaid\s*(.*?)```", text, re.S)
    return (m.group(1) if m else text).strip()

if __name__ == "__main__":
    mcp.run()

Writing mcp_server.py


## 3. Seeded design documents with gold labels
Gold labels follow the same rules the adapters were trained on (see the `analyze()` and labelling code in the training notebooks), so these measure in-distribution seeded-smell detection, as in the proposal. Real human-annotated data is still missing.

In [5]:
STORE = """classDiagram
  class StoreManager {
    -String storeName
    +createOrder()
    +updateOrder()
    +deleteOrder()
    +findCustomer()
    +loadProduct()
    +saveInvoice()
    +validatePayment()
    +sendNotification()
    +calculateTax()
    +renderReport()
    +exportReport()
    +syncInventory()
    +archiveOrder()
    +scheduleDelivery()
  }
  class PaymentGateway {
    +String email
    +String phone
    +double taxRate
    +processPayment(cardNumber: String, expiry: String, cvv: String, amount: double, currency: String, billingAddress: String, shippingAddress: String)
    +validateAndChargeAndNotifyPayment()
  }
  class CustomerRecord {
    +int id
    +String name
    +String address
    +String status
    +Date createdAt
    +Date updatedAt
    +String notes
  }
  class CartService {
    +addItem()
    +removeItem()
    +getTotal()
  }
  class PricingService {
    +priceItem()
  }
  class DiscountService {
    +applyDiscount()
  }
  StoreManager --> CartService
  StoreManager --> PaymentGateway
  StoreManager --> CustomerRecord
  StoreManager --> PricingService
  CartService --> PricingService
  PricingService --> DiscountService
  DiscountService --> CartService"""

LIBRARY = """classDiagram
  class Item {\n +int id\n +String title\n }
  class Media { +int length }
  class Book { +String isbn }
  class Novel { +String genre }
  class Paperback { +int pages }
  class CatalogHub { +searchCatalog() +reserveBook() }
  class Kiosk
  class MobileApp
  class WebPortal
  class Scanner
  class ReportJob
  class BookRepo
  class MemberRepo
  class LoanRepo
  class FineCalculator
  class Mailer
  class LoanDesk {
    +String zipCode
    +int maxRetryCount
    +registerLoan(a: int, b: int, c: String, d: Date, e: Date, f: boolean)
  }
  Media --|> Item
  Book --|> Media
  Novel --|> Book
  Paperback --|> Novel
  Kiosk --> CatalogHub
  MobileApp --> CatalogHub
  WebPortal --> CatalogHub
  Scanner --> CatalogHub
  ReportJob --> CatalogHub
  CatalogHub --> BookRepo
  CatalogHub --> MemberRepo
  CatalogHub --> LoanRepo
  CatalogHub --> FineCalculator
  CatalogHub --> Mailer"""

NOTES = """classDiagram
  class Note {
    +String title
    +String body
    +save()
    +delete()
  }
  class Notebook {
    +addNote(n: Note)
    +removeNote(n: Note)
    +search(q: String)
  }
  Notebook --> Note"""

# a hand-checked diagram can still be wrong: gold = (smell, class) for design, (smell, class, member) for code
SEEDED = {
 "store":   dict(diagram=STORE,   req="Backend of an online store. Customers browse products, pay, and receive invoices.",
    gold_design=[("God Class","StoreManager"),("Data Class","CustomerRecord"),("Cyclic Dependency","CartService"),
                 ("Cyclic Dependency","PricingService"),("Cyclic Dependency","DiscountService")],
    gold_code=[("Long Parameter List","PaymentGateway","processPayment"),("Long Method","PaymentGateway","validateAndChargeAndNotifyPayment"),
               ("Primitive Obsession","PaymentGateway","email"),("Primitive Obsession","PaymentGateway","phone"),
               ("Magic Number Risk","PaymentGateway","taxRate")]),
 "library": dict(diagram=LIBRARY, req="A library system. Members borrow and return books, fines are calculated for late returns, and the catalog can be searched from kiosks, a mobile app and a web portal.",
    gold_design=[("Hub-like Modularization","CatalogHub"),("Deep Hierarchy","Paperback")],
    gold_code=[("Long Parameter List","LoanDesk","registerLoan"),("Primitive Obsession","LoanDesk","zipCode"),("Magic Number Risk","LoanDesk","maxRetryCount")]),
 "notes":   dict(diagram=NOTES,   req="A small note-taking app: create, save, delete and search notes in notebooks.",
    gold_design=[], gold_code=[]),
}
os.makedirs("designs", exist_ok=True)
for pid, s in SEEDED.items(): open(f"designs/{pid}.mmd", "w").write("```mermaid\n" + s["diagram"] + "\n```\n")
print({k: (len(v["gold_design"]), len(v["gold_code"])) for k, v in SEEDED.items()}, "(design, code) gold findings")

{'store': (5, 5), 'library': (2, 3), 'notes': (0, 0)} (design, code) gold findings


## 4. Run both SLM conditions through the MCP server
The client reads each diagram with `read_design_doc`, calls both detectors in `finetuned` and `base` mode, retries once on invalid JSON (as the skills specify), and logs latency and GPU energy per call.

In [6]:
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

TOOL = {"design": "detect_design_smells", "code": "detect_code_smells"}
MODES = ("finetuned", "base")

def diagram_classes(d):
    names = set(re.findall(r"^\s*class\s+(\w+)", d, re.M))
    for a, b in re.findall(r"(\w+)\s*(?:-->|\.\.>|--\|>|<\|--)\s*(\w+)", d): names |= {a, b}
    return names

def parse_findings(raw, kind, classes):
    """-> (items, status, n_dropped). Never invents findings; drops unknown smells / classes not in the diagram."""
    txt = re.sub(r"^```(?:json)?|```$", "", raw.strip(), flags=re.M).strip()
    m = re.search(r"\[.*\]", txt, re.S)
    try: data = json.loads(m.group(0) if m else txt)
    except Exception: return [], "invalid_json", 0
    if not isinstance(data, list): return [], "invalid_json", 0
    items, dropped, seen = [], 0, set()
    for x in data:
        if not (isinstance(x, dict) and x.get("smell") in REF[kind] and x.get("class") in classes): dropped += 1; continue
        k = json.dumps(x, sort_keys=True)
        if k not in seen: seen.add(k); items.append(x)
    return items, "ok", dropped

class PowerSampler:
    """Samples GPU board power (W) with nvidia-smi and integrates to joules. energy_j=None if unavailable."""
    def __init__(self, dt=0.25): self.dt, self.j, self.ok, self.run = dt, 0.0, shutil.which("nvidia-smi") is not None, False
    def _loop(self):
        while self.run:
            try:
                w = float(subprocess.check_output(["nvidia-smi","--query-gpu=power.draw","--format=csv,noheader,nounits"], text=True).split()[0])
                self.j += w * self.dt
            except Exception: pass
            time.sleep(self.dt)
    def __enter__(self):
        self.run = True
        if self.ok: self.t = threading.Thread(target=self._loop, daemon=True); self.t.start()
        return self
    def __exit__(self, *a):
        self.run = False
        if self.ok: self.t.join(1)
    @property
    def energy_j(self): return round(self.j, 1) if self.ok else None

def txt(res): return res.content[0].text if res.content else ""

DET = {}   # DET[pid][mode][kind] -> dict(raw, items, status, dropped, attempts, latency_s, energy_j)
async def detect_all():
    params = StdioServerParameters(command=sys.executable, args=["mcp_server.py"],
                                   env={**os.environ, "SMELL_STUB": "1" if STUB else "0"})
    with open("mcp_server.log", "w") as log:                 # Colab's stderr has no fileno(); give the SDK a real file
        async with stdio_client(params, errlog=log) as (rd, wr):
            async with ClientSession(rd, wr) as s:
                await s.initialize()
                print("MCP tools:", [t.name for t in (await s.list_tools()).tools])
                warm = txt(await s.call_tool("read_design_doc", {"path": "designs/notes.mmd"}))
                for mode in MODES:                            # warm-up: model load must not pollute latency numbers
                    for tool in TOOL.values(): await s.call_tool(tool, {"diagram": warm, "mode": mode})
                for pid in SEEDED:
                    diagram = txt(await s.call_tool("read_design_doc", {"path": f"designs/{pid}.mmd"}))
                    classes = diagram_classes(diagram)
                    DET[pid] = {"diagram": diagram}
                    for mode in MODES:
                        DET[pid][mode] = {}
                        for kind, tool in TOOL.items():
                            t0 = time.perf_counter()
                            with PowerSampler() as ps:
                                raw = txt(await s.call_tool(tool, {"diagram": diagram, "mode": mode}))
                                items, status, dropped = parse_findings(raw, kind, classes); attempts = 1
                                if status != "ok":            # skill step 3: retry once, else report failure
                                    raw = txt(await s.call_tool(tool, {"diagram": diagram, "mode": mode, "retry": True}))
                                    items, status, dropped = parse_findings(raw, kind, classes); attempts = 2
                            DET[pid][mode][kind] = dict(raw=raw, items=items, status=status, dropped=dropped, attempts=attempts,
                                                        latency_s=round(time.perf_counter()-t0, 2), energy_j=ps.energy_j)
                            print(f"{pid:8s} {mode:9s} {kind:6s} {status:12s} findings={len(items)} dropped={dropped} {DET[pid][mode][kind]['latency_s']}s")

await detect_all()

MCP tools: ['detect_design_smells', 'detect_code_smells', 'read_design_doc']
store    finetuned design ok           findings=1 dropped=0 3.13s
store    finetuned code   ok           findings=3 dropped=0 19.04s
store    base      design ok           findings=3 dropped=0 12.28s
store    base      code   ok           findings=9 dropped=0 17.59s
library  finetuned design ok           findings=0 dropped=0 0.84s
library  finetuned code   ok           findings=1 dropped=0 6.08s
library  base      design ok           findings=1 dropped=0 2.53s
library  base      code   ok           findings=1 dropped=0 2.81s
notes    finetuned design ok           findings=0 dropped=0 0.56s
notes    finetuned code   ok           findings=0 dropped=0 0.57s
notes    base      design ok           findings=1 dropped=0 2.85s
notes    base      code   ok           findings=2 dropped=0 5.18s


## 5. Detection quality (precision / recall on the seeded diagrams)
Only the **trained** smells are scored. Findings of the other skill-listed smells are counted separately (no gold exists for them).

In [7]:
def fkey(kind, x): return (x["smell"], x["class"]) if kind == "design" else (x["smell"], x["class"], x.get("method") or x.get("attribute"))
TRAINED = {k: set(REF[k]) - UNSEEN[k] for k in REF}

rows = []
for mode in MODES:
    for kind in ("design", "code"):
        tp = fp = fn = extra = bad = 0
        for pid, s in SEEDED.items():
            r = DET[pid][mode][kind]; bad += r["status"] != "ok"
            gold = set(s[f"gold_{kind}"])
            pred = {fkey(kind, x) for x in r["items"] if x["smell"] in TRAINED[kind]}
            extra += sum(x["smell"] not in TRAINED[kind] for x in r["items"])
            tp += len(pred & gold); fp += len(pred - gold); fn += len(gold - pred)
        P = tp/max(tp+fp,1); R = tp/max(tp+fn,1); F = 2*P*R/max(P+R,1e-9)
        rows.append(dict(condition="A fine-tuned" if mode=="finetuned" else "B untrained", detector=kind, TP=tp, FP=fp, FN=fn,
                         precision=round(P,2), recall=round(R,2), F1=round(F,2), other_smells_flagged=extra, invalid_json=bad))
DETECT_DF = pd.DataFrame(rows); print(DETECT_DF.to_markdown(index=False))
DETECT_DF.to_csv(f"{ART}/detection_quality.csv", index=False)

| condition    | detector   |   TP |   FP |   FN |   precision |   recall |   F1 |   other_smells_flagged |   invalid_json |
|:-------------|:-----------|-----:|-----:|-----:|------------:|---------:|-----:|-----------------------:|---------------:|
| A fine-tuned | design     |    1 |    0 |    6 |         1   |     0.14 | 0.25 |                      0 |              0 |
| A fine-tuned | code       |    4 |    0 |    4 |         1   |     0.5  | 0.67 |                      0 |              0 |
| B untrained  | design     |    1 |    4 |    6 |         0.2 |     0.14 | 0.17 |                      0 |              0 |
| B untrained  | code       |    0 |    0 |    8 |         0   |     0    | 0    |                     12 |              0 |


## 6. Build the DAD (one per project per SLM condition)
Follows the skills: IDs `D1..`/`C1..`, design rows sorted high to low, explanations/fixes from the skill tables, `[untrained probe]` prefix for skill-listed unseen smells, severity for code smells by the skill's rule.

In [8]:
SEV_ORDER = {"high": 0, "medium": 1, "low": 2}
esc = lambda s: str(s).replace("|", "\\|").replace("\n", " ")

def build_dad(design_f, code_f, source):
    d_rows, c_grouped = {}, {}
    for f in design_f:
        k = (f["smell"], f["class"])
        if k not in d_rows:
            meaning, fix = DESIGN_REF[f["smell"]]
            if f["smell"] in DESIGN_UNSEEN: meaning = "[untrained probe] " + meaning
            d_rows[k] = dict(smell=f["smell"], loc=f["class"], sev=f.get("severity") if f.get("severity") in SEV_ORDER else "medium", why=meaning, fix=fix)
    for f in code_f:
        k = (f["smell"], f["class"])
        member = f.get("method") or f.get("attribute")
        meaning, fix = CODE_REF[f["smell"]]
        why = f.get("reason") or meaning
        if f["smell"] in CODE_UNSEEN: why = "[untrained probe] " + why
        r = c_grouped.setdefault(k, dict(smell=f["smell"], locs=[], sev=CODE_SEV.get(f["smell"], "low"), why=why, fix=fix))
        r["locs"].append(f"{f['class']}.{member}" if member else f["class"])
    D = sorted(d_rows.values(), key=lambda r: (SEV_ORDER[r["sev"]], r["smell"], r["loc"]))
    C = sorted(c_grouped.values(), key=lambda r: (SEV_ORDER[r["sev"]], r["smell"], r["locs"][0]))
    lines = ["# Design Advisory Document", f"Source: {source}", "",
             "| ID | Smell | Location | Severity | Explanation | Suggested fix |", "|----|-------|----------|----------|-------------|---------------|"]
    lines += [f"| D{i} | {esc(r['smell'])} | {esc(r['loc'])} | {r['sev']} | {esc(r['why'])} | {esc(r['fix'])} |" for i, r in enumerate(D, 1)]
    lines += [f"| C{i} | {esc(r['smell'])} | {esc(', '.join(r['locs']))} | {r['sev']} | {esc(r['why'])} | {esc(r['fix'])} |" for i, r in enumerate(C, 1)]
    if not D and not C: lines.append("| - | No smells detected | - | - | - | - |")
    return "\n".join(lines)

DAD = {"A": {}, "B": {}}
for pid in SEEDED:
    for tag, mode in (("A", "finetuned"), ("B", "base")):
        DAD[tag][pid] = build_dad(DET[pid][mode]["design"]["items"], DET[pid][mode]["code"]["items"], f"designs/{pid}.mmd")
        open(f"{ART}/DAD_{tag}_{pid}.md", "w").write(DAD[tag][pid])
print(DAD["A"]["store"]); print(); print(DAD["B"]["store"])

# Design Advisory Document
Source: designs/store.mmd

| ID | Smell | Location | Severity | Explanation | Suggested fix |
|----|-------|----------|----------|-------------|---------------|
| D1 | Data Class | CustomerRecord | low | A class with attributes (6+ in training) but no behaviour | Move related behaviour into the class (Move Method) or merge it into its main user |
| C1 | Long Method | PaymentGateway.validateAndChargeAndNotifyPayment | medium | Method name combines 3 or more verbs. | Split into single-purpose methods; extract helpers |
| C2 | Primitive Obsession | PaymentGateway.email, PaymentGateway.phone | low | Domain concept represented as a raw String. | Create small value classes (`Email`, `Money`, `ZipCode`) with validation |

# Design Advisory Document
Source: designs/store.mmd

| ID | Smell | Location | Severity | Explanation | Suggested fix |
|----|-------|----------|----------|-------------|---------------|
| D1 | Cyclic Dependency | StoreManager | high | A dependenc

## 7. Code generation: baseline vs A vs B
Identical prompt scaffolding; only the DAD block differs. Gemini thinking is turned off so the output budget is not consumed by reasoning.

In [9]:
from google import genai
from google.genai import types
if STUB: userdata = None
else: from google.colab import userdata
client = genai.Client(api_key="stub" if STUB else userdata.get("MidSub"))

def gen(prompt, tries=4):
    for a in range(tries):
        try:
            t0 = time.perf_counter()
            r = client.models.generate_content(model=MODEL, contents=prompt, config=types.GenerateContentConfig(
                temperature=0.2, max_output_tokens=8192, thinking_config=types.ThinkingConfig(thinking_budget=0)))
            u = r.usage_metadata
            return (r.text or ""), time.perf_counter()-t0, u.prompt_token_count, u.candidates_token_count
        except Exception as e:
            print("Gemini error, retrying:", str(e)[:120]); time.sleep(8 * (a + 1))
    return "", 0.0, 0, 0

def extract_py(t):
    m = re.search(r"```python\s*(.*?)```", t, re.S) or re.search(r"```\s*(.*?)```", t, re.S)
    return (m.group(1) if m else t).strip()

def make_prompt(pid, dad=None):
    s = SEEDED[pid]
    p = f"You are a senior Python engineer.\nHere is the design document (Mermaid) and requirements:\n{s['req']}\n{s['diagram']}\n\n"
    if dad: p += f"Here is the Design Advisory Document (DAD) containing identified smells:\n{dad}\n\n"
    p += "Task: Generate the complete Python code."
    if dad: p += " Use the DAD to improve the code and avoid the listed flaws."
    return p + "\nOutput only the Python code in a ```python block."

COND = {"baseline": None, "A_finetuned_dad": "A", "B_untrained_dad": "B"}
GEN = []   # one record per generation
for pid in SEEDED:
    for run in range(N_RUNS):
        for cond, tag in COND.items():
            txt_, dt, tin, tout = gen(make_prompt(pid, DAD[tag][pid] if tag else None))
            code_ = extract_py(txt_)
            path = f"{ART}/{pid}_{cond}_r{run}.py"; open(path, "w").write(code_)
            mode = {"A": "finetuned", "B": "base"}.get(tag)
            slm_lat = sum(DET[pid][mode][k]["latency_s"] for k in TOOL) if mode else 0.0
            slm_j = [DET[pid][mode][k]["energy_j"] for k in TOOL] if mode else [0.0]
            GEN.append(dict(project=pid, condition=cond, run=run, path=path, llm_in_tok=tin, llm_out_tok=tout,
                            llm_latency_s=round(dt,2), slm_latency_s=slm_lat,
                            slm_energy_j=None if None in slm_j else round(sum(slm_j),1)))
            print(pid, cond, run, f"in={tin} out={tout} {dt:.1f}s")
            time.sleep(1 if STUB else 6)    # stay under free-tier rate limits

store baseline 0 in=379 out=737 1.7s
store A_finetuned_dad 0 in=582 out=824 2.5s
store B_untrained_dad 0 in=721 out=866 2.7s
store baseline 1 in=379 out=811 2.2s
store A_finetuned_dad 1 in=582 out=852 2.4s
store B_untrained_dad 1 in=721 out=797 2.3s
library baseline 0 in=341 out=781 1.9s
library A_finetuned_dad 0 in=454 out=641 1.8s
library B_untrained_dad 0 in=507 out=877 2.2s
library baseline 1 in=341 out=781 1.9s
library A_finetuned_dad 1 in=454 out=747 15.9s
library B_untrained_dad 1 in=507 out=891 2.1s
notes baseline 0 in=134 out=352 1.4s
notes A_finetuned_dad 0 in=221 out=482 1.5s
notes B_untrained_dad 0 in=303 out=501 1.6s
notes baseline 1 in=134 out=352 1.2s
notes A_finetuned_dad 1 in=221 out=499 4.9s
notes B_untrained_dad 1 in=303 out=391 3.3s


## 8. Independent quality measurement
None of this uses the SLMs. **PyExamine / DPy (named in the proposal) are not wired in here**, so these are stand-in proxies: pylint design checks + AST/networkx metrics. Swap in PyExamine/DPy for the final report.

In [10]:
def ast_metrics(src, diag_cls):
    try: tree = ast.parse(src)
    except SyntaxError: return dict(compiles=False)
    classes = {n.name: n for n in ast.walk(tree) if isinstance(n, ast.ClassDef)}
    funcs = [n for n in ast.walk(tree) if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))]
    def nparams(f):
        a = f.args; return len([x for x in a.posonlyargs + a.args + a.kwonlyargs if x.arg not in ("self", "cls")]) + bool(a.vararg) + bool(a.kwarg)
    g = nx.DiGraph(); g.add_nodes_from(classes)
    for name, c in classes.items():
        for n in ast.walk(c):
            ref = n.id if isinstance(n, ast.Name) else (n.value if isinstance(n, ast.Constant) and isinstance(n.value, str) else None)
            if ref in classes and ref != name: g.add_edge(name, ref)
    cyc = sum(len(c) for c in nx.strongly_connected_components(g) if len(c) > 1)
    magic = sum(1 for f in funcs for n in ast.walk(f)
                if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)) and not isinstance(n.value, bool) and n.value not in (0, 1, -1))
    god = sum(len([m for m in c.body if isinstance(m, (ast.FunctionDef, ast.AsyncFunctionDef)) and not m.name.startswith("_")]) >= 12 for c in classes.values())
    loc = max(sum(1 for l in src.splitlines() if l.strip()), 1)
    d = dict(compiles=True, loc=loc, classes=len(classes), max_params=max((nparams(f) for f in funcs), default=0),
             too_many_args=sum(nparams(f) > 5 for f in funcs), long_methods=sum((f.end_lineno - f.lineno + 1) > 30 for f in funcs),
             god_classes=god, classes_in_cycles=cyc, magic_numbers=magic,
             diagram_coverage=round(len([c for c in diag_cls if c in classes]) / max(len(diag_cls), 1), 2))
    d["ast_smells_per_kloc"] = round(1000 * (d["too_many_args"] + d["long_methods"] + d["god_classes"] + d["classes_in_cycles"] + d["magic_numbers"]) / loc, 1)
    return d

def pylint_count(path):
    r = subprocess.run([sys.executable, "-m", "pylint", path, "--disable=all", "--enable=R0913,R0902,R0904,R0914,R0915,R0917", "--output-format=json"],
                       capture_output=True, text=True)
    try: return sum(1 for m in json.loads(r.stdout or "[]") if str(m.get("message-id", "")).startswith("R09"))
    except Exception: return None

for rec in GEN:
    src = open(rec["path"]).read()
    rec.update(ast_metrics(src, diagram_classes(SEEDED[rec["project"]]["diagram"])))
    rec["pylint_design_msgs"] = pylint_count(rec["path"]) if rec.get("compiles") else None
GEN_DF = pd.DataFrame(GEN); GEN_DF.to_csv(f"{ART}/generation_runs.csv", index=False)
print(GEN_DF.drop(columns="path").to_markdown(index=False))

| project   | condition       |   run |   llm_in_tok |   llm_out_tok |   llm_latency_s |   slm_latency_s |   slm_energy_j | compiles   |   loc |   classes |   max_params |   too_many_args |   long_methods |   god_classes |   classes_in_cycles |   magic_numbers |   diagram_coverage |   ast_smells_per_kloc |   pylint_design_msgs |
|:----------|:----------------|------:|-------------:|--------------:|----------------:|----------------:|---------------:|:-----------|------:|----------:|-------------:|----------------:|---------------:|--------------:|--------------------:|----------------:|-------------------:|----------------------:|---------------------:|
| store     | baseline        |     0 |          379 |           737 |            1.74 |            0    |            0   | True       |    60 |         6 |            7 |               1 |              0 |             1 |                   0 |               3 |                  1 |                  83.3 |                    2 |
| store

## 9. Summary and export

In [11]:
num = ["llm_in_tok","llm_out_tok","llm_latency_s","slm_latency_s","slm_energy_j","compiles","diagram_coverage","max_params","too_many_args",
       "long_methods","god_classes","classes_in_cycles","magic_numbers","ast_smells_per_kloc","pylint_design_msgs","loc"]
SUM = GEN_DF.assign(compiles=GEN_DF["compiles"].astype(float)).groupby("condition")[[c for c in num if c in GEN_DF]].mean().round(2)
SUM["total_latency_s"] = (SUM["llm_latency_s"] + SUM["slm_latency_s"]).round(2)
print("Mean per generation, over projects x runs (slm_* = SLM cost of building that condition's DAD)\n"); print(SUM.T.to_markdown())
SUM.to_csv(f"{ART}/summary.csv")
json.dump({p: {m: {k: {x: v for x, v in DET[p][m][k].items()} for k in TOOL} for m in MODES} for p in SEEDED},
          open(f"{ART}/slm_raw_outputs.json", "w"), indent=1)
shutil.make_archive("midsub_artifacts", "zip", ART)
if not STUB:
    from google.colab import files; files.download("midsub_artifacts.zip")

Mean per generation, over projects x runs (slm_* = SLM cost of building that condition's DAD)

|                     |   A_finetuned_dad |   B_untrained_dad |   baseline |
|:--------------------|------------------:|------------------:|-----------:|
| llm_in_tok          |            419    |            510.33 |     284.67 |
| llm_out_tok         |            674.17 |            720.5  |     635.67 |
| llm_latency_s       |              4.84 |              2.36 |       1.72 |
| slm_latency_s       |             10.07 |             14.41 |       0    |
| slm_energy_j        |            324.63 |            669.93 |       0    |
| compiles            |              1    |              1    |       1    |
| diagram_coverage    |              1    |              1    |       1    |
| max_params          |              3.83 |              3.67 |       5    |
| too_many_args       |              0.5  |              1    |       1    |
| long_methods        |              0    |              0

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>